In [50]:
import numpy as np
import pandas as pd
import os
import sys

### LOAD Data

In [51]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader, DirectoryLoader
docs = DirectoryLoader(path=r'../sources',
                       glob="*.pdf",
                       loader_cls=PyPDFLoader,
                       show_progress=True).load()
for i in docs:
    print(len(i.page_content),end=" ")

#text_docs = [ doc.page_content for doc in docs]

100%|██████████| 2/2 [00:01<00:00,  1.65it/s]

0 0 2666 3929 5310 4744 4148 1722 612 309 4145 2684 3972 7230 4142 2982 1451 3549 4293 2012 5698 2147 6579 5849 5908 5987 6296 5680 8024 6593 0 0 384 0 1313 2035 1987 1959 1991 2050 2012 2137 2018 2063 2036 1963 1921 2017 2099 1977 1881 435 1989 2015 1715 

### Split Data in Chunks

In [52]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def chunk_split(docs,chunk_size,chunk_overlap):
    chuck_obj = RecursiveCharacterTextSplitter(chunk_size=chunk_size, 
                                               chunk_overlap=chunk_overlap, 
                                               separators=['\n\n','\n','',' '],
                                               add_start_index=True)
    return chuck_obj.split_documents(docs)

In [53]:
doc_chunks = chunk_split(docs,chunk_size=1000, chunk_overlap=200)
for i in doc_chunks:
    print(len(i.page_content),end=" ")

996 959 973 279 942 994 971 965 723 940 980 957 996 953 965 592 946 992 985 977 994 665 995 983 963 997 898 976 865 612 309 999 945 970 994 874 976 971 980 285 968 982 999 966 806 956 954 996 972 995 959 988 962 708 946 954 945 990 995 961 963 997 546 992 639 971 888 981 972 351 948 948 944 944 970 449 970 977 417 988 997 944 942 981 976 849 960 966 594 951 976 948 964 997 974 993 957 969 951 979 952 942 953 986 363 993 994 950 959 955 960 978 346 990 921 991 946 955 986 993 408 977 956 944 999 950 990 962 769 968 977 987 975 975 974 851 1000 955 962 993 985 966 965 973 993 875 967 970 996 987 982 964 942 928 384 971 522 998 914 496 957 965 426 952 973 391 973 984 379 987 965 463 993 946 398 963 990 540 999 976 416 974 988 457 955 932 509 994 981 345 963 970 333 962 981 428 967 978 457 981 989 364 937 966 316 435 991 981 357 954 987 369 981 888 

### Embed Chunks

In [54]:
from langchain_text_splitters.sentence_transformers import SentenceTransformer
embedding_model = "BAAI/bge-base-en-v1.5"

class Embedding_Model:
    def __init__(self):
        self.embedding_model_name = "BAAI/bge-base-en-v1.5"
        self.model = SentenceTransformer(model_name_or_path=self.embedding_model_name)
    def generateEncoding(self, sens):
        return self.model.encode(sens)

embedModel = Embedding_Model()
text_li = [chunk.page_content for chunk in doc_chunks]
embed_li = embedModel.generateEncoding(text_li)

In [57]:
import chromadb
from chromadb.config import Settings
import uuid
from sklearn.metrics.pairwise import cosine_similarity

class VectorDb():

    def __init__(self, storage_path):
        self.client = chromadb.PersistentClient(path=storage_path)

    def get_db(self,collection_name):
        if(self.client.list_collections() and self.client.list_collections()[0].name==collection_name):
            self.client.delete_collection(collection_name)
        self.db = self.client.get_or_create_collection(collection_name,metadata={'desc':'pdf'})

    def add_rows(self,collection_name,embed_li, doc_chunks):
        self.get_db(collection_name)
        total_chunks = len(embed_li)
        text_chunks = [doc.page_content for doc in doc_chunks]
        metadata = [doc.metadata for doc in doc_chunks]
        ids = [f"doc_{uuid.uuid4().hex[:8]}_{i}" for i in range(total_chunks)]
        self.db.add(ids=ids,
                    embeddings=embed_li,
                    metadatas=metadata,
                    documents=text_chunks)
        print("Total record added->",self.db.count())
    
    def get_contexts(self,embedded_queries,n_results,confidence_thresold):
        results = self.db.query(query_embeddings=embedded_queries,n_results=n_results)
        text_chunks = results['documents'][0]
        context_st = ''
        citation = ''
        for i, dist in enumerate(results['distances'][0]):
            similarity = 1-dist
            if similarity>confidence_thresold:
                context_st += '/n/n' + text_chunks[i]
                metadata = results['metadatas'][0][i]
                start_index = metadata['start_index']
                source = metadata['source']
                page = metadata['page']
                confidence = similarity
                citation += f'source:{source} page:{page} start_index:{start_index} confidence {confidence}\n'

        return context_st, citation


prompt="what was einstein's weakness?"
vectorDb = VectorDb(storage_path='../vector_db')
vectorDb.add_rows('pdf_collections',embed_li, doc_chunks)
embedded_prompt = embedModel.generateEncoding([prompt])[0]
print(embedded_prompt.shape)
contexts, citation = vectorDb.get_contexts(embedded_prompt,5,0.1)

Total record added-> 214
(768,)


### Run Model

In [58]:
from langchain_groq import ChatGroq
import dotenv
dotenv.load_dotenv()
api = os.getenv('GROQ_API_KEY')
query = f"""Using the following context answer the question correctly.
                context={contexts}
                question={prompt}
                answer="""
model = ChatGroq(model='llama-3.1-8b-instant',temperature=0.1,max_tokens=1024,api_key=api)
print("answer->",model.invoke([query]).content)
print("CITATIONS ->\n",citation)

answer-> The text does not explicitly mention Einstein's weakness. However, it does mention that Einstein's university record was "unpromising" and that he was considered to have had an "unpromising university record." This suggests that Einstein may have struggled academically or had a lackluster academic performance during his university years.
CITATIONS ->
 source:..\sources\einstein-albert.pdf page:10 start_index:807 confidence 0.35029423236846924
source:..\sources\einstein-albert.pdf page:9 start_index:1597 confidence 0.34449613094329834
source:..\sources\einstein-albert.pdf page:7 start_index:811 confidence 0.34029674530029297
source:..\sources\einstein-albert.pdf page:16 start_index:787 confidence 0.3156886100769043
source:..\sources\einstein-albert.pdf page:10 start_index:0 confidence 0.3124140501022339

